# <font color='orange'>**RapidRelief AI — Sprint 2 & 3: Transferencia de Aprendizaje**</font>

**Metodología:** MobileNetV2/EfficientNetB0 + `include_top=False` + cabeza personalizada + fine-tuning  
**Dataset:** Clothing Small (fotos reales de prendas)  
**Meta:** `val_accuracy ≥ 0.90`

| Versión | Backbone | Loss | Val acc |
|---------|----------|------|---------|
| v2 | MobileNetV2 | CrossEntropy | 0.8827 |
| v3 | MobileNetV2 | CrossEntropy + cw | 0.8886 |
| v4 | MobileNetV2 | Focal Loss + cw | 0.8886 |
| **v5** | **EfficientNetB0** | **CrossEntropy + cw** | **→ ?** |

**Cambios v5:** backbone EfficientNetB0 · misma cabeza GAP → Dense(512) → Dropout(0.4) · Adam lr=1e-3 · CrossEntropy + class weights · fine-tuning últimas 60 capas

## 1. Importar el modelo de interés

Mismo patrón que en clase — `include_top=False` + `TF_USE_LEGACY_KERAS`.  
**v5:** backbone cambiado a **EfficientNetB0** (mismo tamaño ~5 MB, +2–3% accuracy sobre MobileNetV2 en clasificación de ropa).

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
%matplotlib inline

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Rutas del proyecto
BASE_DIR    = '/content/drive/MyDrive/RapidReliefAI'
CLOTH_TRAIN = BASE_DIR + '/Datasets/clothing_small/train'
CLOTH_VAL   = BASE_DIR + '/Datasets/clothing_small/validation'
CLOTH_TEST  = BASE_DIR + '/Datasets/clothing_small/test'
MODELS_DIR  = BASE_DIR + '/Models'

# Parametros — mismos que en clase
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
N_CLASES   = 10

# Etiquetas finales (orden alfabetico = orden que asigna flow_from_directory)
CLASES = ['dress', 'hat', 'longsleeve', 'outwear', 'pants',
          'shirt', 'shoes', 'shorts', 'skirt', 't-shirt']
print('Configuracion lista')
print(f'Clases ({N_CLASES}):', CLASES)

## 2. Importar la ruta del DATASET

Mismo patrón que en clase: `ImageDataGenerator` + `flow_from_directory` + `preprocessing_function=preprocess_input`.

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=15.0,
    zoom_range=0.20,
    horizontal_flip=True,
    brightness_range=[0.80, 1.20],
    fill_mode='reflect'
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

print('Cargando datos...')
train_data = train_datagen.flow_from_directory(
    CLOTH_TRAIN,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=42
)

val_data = val_datagen.flow_from_directory(
    CLOTH_VAL,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=42
)

test_data = val_datagen.flow_from_directory(
    CLOTH_TEST,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
    seed=42
)

In [ ]:
# Verificar forma del batch — igual que en clase
imgs, labels = next(train_data)
print(f'Forma imagenes: {imgs.shape}')   # (32, 224, 224, 3)
print(f'Forma etiquetas: {labels.shape}') # (32, 10)
print('Indice de clases:', train_data.class_indices)

In [ ]:
# Visualizar batch — misma funcion que en clase
def plotImages(images_arr):
    fig, axes = plt.subplots(1, 10, figsize=(20, 20))
    axes = axes.flatten()
    for img, ax in zip(images_arr, axes):
        # Revertir normalización torch (EfficientNetB0: mean/std de ImageNet)
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        img_vis = np.clip(img * std + mean, 0, 1)
        ax.imshow(img_vis)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

plotImages(imgs)
print(labels[:10])
# Etiquetas: 0=dress 1=hat 2=longsleeve 3=outwear 4=pants 5=shirt 6=shoes 7=shorts 8=skirt 9=t-shirt

## 3. Cargar el modelo pre-entrenado

Igual que en clase: `include_top=False` para no cargar las capas densas de ImageNet,  
y `trainable = False` para congelar todos los pesos del modelo base.

In [ ]:
# EfficientNetB0: mismo patrón que clase — include_top=False, weights='imagenet'
# preprocess_input de efficientnet normaliza con media/std de ImageNet (modo torch)
pre_trained_model = EfficientNetB0(
    input_shape=IMAGE_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)

In [ ]:
# Se debe congelar el modelo base — no se ajustaran los pesos del modelo pre-entrenado
pre_trained_model.trainable = False

### <font color='darkviolet'>**Cabeza personalizada — API Funcional de Keras**</font>

Mismo patrón que en clase — API funcional de Keras:
- `GlobalAveragePooling2D` promedia el tensor de salida de EfficientNetB0 a 1 280 features
- `Dense(512, relu)` + `Dropout(0.4)` para regularización
- `Dense(10, softmax)` — salida multi-clase

In [ ]:
pre_trained_model.output

In [ ]:
pre_trained_model.input

In [ ]:
pre_trained_model.summary()

In [ ]:
# GAP → Dense(512) → Dropout → Dense(10)
x = GlobalAveragePooling2D()(pre_trained_model.output)
x = Dense(512, activation='relu')(x)
x = Dropout(0.4)(x)
output = Dense(N_CLASES, activation='softmax')(x)

In [ ]:
modelo = Model(inputs=pre_trained_model.input, outputs=output)
modelo.summary()

In [ ]:
modelo.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=1e-3),
    metrics=['accuracy']
)
print('Modelo compilado — Phase A | base congelada | Adam lr=1e-3 | CrossEntropy')

## 4. Fase A — Entrenar la cabeza (base congelada)

Se entrena solo la cabeza nueva con la base MobileNetV2 completamente congelada.  
`class_weight` compensa el desbalance entre clases (hat/skirt: 12 imgs vs shoes: 73).

In [ ]:
# Class weights — compensar desbalance entre clases
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(N_CLASES),
    y=train_data.classes
)
cw = dict(enumerate(class_weights_arr))

# Ajuste basado en análisis de matriz de confusión v3:
# shirt (idx=5): F1=0.53, precision=0.48 — confusión severa con t-shirt y longsleeve
# longsleeve (idx=2): recall=0.74 — falsos negativos predichos como shirt
# outwear (idx=3): recall=0.74 — subestimado, confusión con longsleeve
cw[2] *= 1.5   # longsleeve
cw[3] *= 1.5   # outwear
cw[5] *= 2.5   # shirt

print('Class weights ajustados (v4):')
for k, v in cw.items():
    print(f'  {CLASES[k]:<12} {v:.3f}')

os.makedirs(MODELS_DIR, exist_ok=True)

callbacks_A = [
    ModelCheckpoint(
        MODELS_DIR + '/rapidrelief_phaseA_best.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=10,          # era 6 — más margen para superar mesetas temporales
        min_delta=0.001,      # solo contar mejoras reales (>0.1%), ignorar ruido
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,           # era 3
        min_lr=1e-6,
        verbose=1
    ),
]

history_A = modelo.fit(
    train_data,
    validation_data=val_data,
    epochs=40,                # era 25 — más margen para que focal loss converja
    callbacks=callbacks_A,
    class_weight=cw
)

## 5. Guardar modelo Phase A

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)
modelo.save(MODELS_DIR + '/rapidrelief_mobilenetv2_phaseA.keras')
print('Modelo Phase A guardado:', MODELS_DIR + '/rapidrelief_mobilenetv2_phaseA.keras')

## 6. Curvas de entrenamiento — Phase A

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history_A.history['accuracy'], label='Train')
plt.plot(history_A.history['val_accuracy'], 'r--', label='Validación')
plt.title('Phase A — Exactitud')
plt.ylabel('Exactitud')
plt.xlabel('Épocas')
plt.axhline(0.90, color='green', linestyle=':', linewidth=1, label='KPI 90%')
plt.legend()
plt.tight_layout()
plt.show()

gap = max(history_A.history['accuracy']) - max(history_A.history['val_accuracy'])
print(f'Val accuracy máx: {max(history_A.history["val_accuracy"]):.4f}')
print(f'Brecha train-val: {gap:.4f} (objetivo < 0.05)')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history_A.history['loss'], label='Train')
plt.plot(history_A.history['val_loss'], 'r--', label='Validación')
plt.title('Phase A — Pérdida')
plt.ylabel('Loss')
plt.xlabel('Épocas')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Fase B — Fine-tuning (últimas 60 capas de EfficientNetB0)

Con la cabeza ya entrenada, descongelamos las **últimas 60 capas** del backbone y ajustamos con LR más bajo (`1e-5`) para refinar representaciones sin destruir los pesos de ImageNet.

In [ ]:
pre_trained_model.trainable = True
for layer in pre_trained_model.layers[:-60]:
    layer.trainable = False

trainable_count = sum(1 for l in pre_trained_model.layers if l.trainable)
print(f'Capas entrenables en base: {trainable_count} de {len(pre_trained_model.layers)}')

modelo.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=1e-5),
    metrics=['accuracy']
)
print('Modelo recompilado — Phase B | 60 capas | Adam lr=1e-5 | CrossEntropy')

callbacks_B = [
    ModelCheckpoint(
        MODELS_DIR + '/rapidrelief_phaseB_best.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=20,
        min_delta=0.001,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
]

history_B = modelo.fit(
    train_data,
    validation_data=val_data,
    epochs=60,
    callbacks=callbacks_B,
    class_weight=cw
)

In [ ]:
modelo.save(MODELS_DIR + '/rapidrelief_efficientnetb0_v5.keras')
print('Modelo final (Phase B) guardado:', MODELS_DIR + '/rapidrelief_efficientnetb0_v5.keras')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history_B.history['accuracy'], label='Train')
plt.plot(history_B.history['val_accuracy'], 'r--', label='Validación')
plt.title('Phase B — Exactitud (fine-tuning)')
plt.ylabel('Exactitud')
plt.xlabel('Épocas')
plt.axhline(0.90, color='green', linestyle=':', linewidth=1, label='KPI 90%')
plt.legend()
plt.tight_layout()
plt.show()

gap_B = max(history_B.history['accuracy']) - max(history_B.history['val_accuracy'])
print(f'Val accuracy Phase B máx: {max(history_B.history["val_accuracy"]):.4f}')
print(f'Brecha train-val Phase B: {gap_B:.4f}')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history_B.history['loss'], label='Train')
plt.plot(history_B.history['val_loss'], 'r--', label='Validación')
plt.title('Phase B — Pérdida (fine-tuning)')
plt.ylabel('Loss')
plt.xlabel('Épocas')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Evaluación — Matriz de Confusión y Reporte por Clase

Herramienta clave para identificar solapamientos entre clases similares (ej. shirt vs t-shirt).

In [ ]:
# TTA — Test-Time Augmentation
# Literatura: +1.5–2% accuracy en clases con bajo recall (longsleeve, outwear)
# Principio: prendas simétricas (camisas, pantalones) son invariantes al flip horizontal
# Soft voting: promediar logits es más estable que votar la clase más frecuente

test_datagen_flip = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True
)
test_data_flip = test_datagen_flip.flow_from_directory(
    CLOTH_TEST,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
    seed=42
)

# Predicciones originales
test_data.reset()
preds_orig = modelo.predict(test_data, verbose=1)

# Predicciones con flip horizontal
test_data_flip.reset()
preds_flip = modelo.predict(test_data_flip, verbose=1)

# Soft voting: promedio de logits
predicciones_tta = (preds_orig + preds_flip) / 2.0

y_pred     = np.argmax(preds_orig, axis=1)
y_pred_tta = np.argmax(predicciones_tta, axis=1)
y_true     = test_data.classes

acc_base = np.mean(y_pred == y_true)
acc_tta  = np.mean(y_pred_tta == y_true)
print(f'Accuracy sin TTA:  {acc_base:.4f}')
print(f'Accuracy con TTA:  {acc_tta:.4f}  (Δ = {acc_tta - acc_base:+.4f})')
print(f'Total predicciones: {len(y_pred_tta)}')

# Usar TTA para el reporte final
predicciones = predicciones_tta

In [ ]:
# Matriz de Confusion con Seaborn (igual que en brief.md)
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12, 9))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASES,
    yticklabels=CLASES
)
plt.title('Matriz de Confusion — RapidRelief AI (MobileNetV2)', fontsize=13)
plt.ylabel('Etiqueta real')
plt.xlabel('Etiqueta predicha')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Reporte completo por clase (precision, recall, F1)
print('Reporte de clasificacion por clase:')
print(classification_report(y_true, y_pred, target_names=CLASES))

## 9. Inferencia con imágenes individuales

Verifica predicciones sobre imágenes reales del conjunto de test.

In [ ]:
from tensorflow.keras.preprocessing import image

def predecir_prenda(img_path):
    img = image.load_img(img_path, target_size=IMAGE_SIZE)
    X = image.img_to_array(img)
    X = np.expand_dims(X, axis=0)
    X = preprocess_input(X)
    preds = modelo.predict(X, verbose=0)
    clase_idx = np.argmax(preds, axis=1)[0]
    clase_nombre = CLASES[clase_idx]
    confianza = preds[0][clase_idx]

    print(f'Prediccion: {clase_nombre} (confianza: {confianza:.2%})')
    print(f'Vector de probabilidades: {preds[0].round(3)}')

    plt.imshow(img)
    plt.title(f'Prediccion: {clase_nombre} ({confianza:.1%})')
    plt.axis('off')
    plt.show()
    return clase_nombre

In [ ]:
# Probar con una imagen del conjunto de test
# Modificar img_path con cualquier imagen real de tus carpetas de test
img_path = CLOTH_TEST + '/dress/' + os.listdir(CLOTH_TEST + '/dress/')[0]
predecir_prenda(img_path)

## 10. Exportar a TF Lite (para app móvil)

Cuantización INT8 — reduce el modelo de ~14 MB a ~4 MB sin pérdida significativa de accuracy.

In [ ]:
import tensorflow as tf

# Conjunto representativo para calibración INT8 (100 batches del test set)
def representative_dataset():
    test_data.reset()
    count = 0
    for imgs, _ in test_data:
        for img in imgs:
            yield [np.expand_dims(img.astype(np.float32), axis=0)]
            count += 1
            if count >= 100:
                return

converter = tf.lite.TFLiteConverter.from_keras_model(modelo)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_model = converter.convert()

tflite_path = MODELS_DIR + '/rapidrelief_model.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f'Modelo TFLite INT8 guardado: {tflite_path}')
print(f'Tamaño: {size_mb:.1f} MB  (objetivo: ≤ 6 MB)')

In [ ]:
# Guardar archivo de etiquetas para la app
labels_path = MODELS_DIR + '/labels.txt'
with open(labels_path, 'w') as f:
    for clase in CLASES:
        f.write(clase + '\n')

print('Archivo de etiquetas guardado:')
for i, clase in enumerate(CLASES):
    print(f'  {i}: {clase}')

## 11. Resumen final

In [ ]:
val_acc_A   = max(history_A.history['val_accuracy'])
val_acc_B   = max(history_B.history['val_accuracy'])
train_acc_B = max(history_B.history['accuracy'])
gap         = train_acc_B - val_acc_B

print('=== RESUMEN FINAL — v5 ===')
print(f'Arquitectura:      EfficientNetB0 → GAP → Dense(512,relu) → Dropout(0.4) → Dense(10,softmax)')
print(f'Optimizador A:     Adam lr=1e-3  |  Optimizador B: Adam lr=1e-5')
print(f'Loss:              categorical_crossentropy + class weights')
print(f'Augmentation:      rotation=20, shift=0.15, shear=15, zoom=0.20, brightness=[0.80,1.20], fill=reflect')
print(f'Class weights:     balanced + longsleeve×1.5 + outwear×1.5 + shirt×2.5')
print(f'Fine-tuning:       últimas 60 capas | patience_B=20 | min_delta=0.001 | epochs_B≤60')
print(f'Val accuracy Phase A: {val_acc_A:.4f}')
print(f'Val accuracy Phase B: {val_acc_B:.4f}')
print(f'Brecha train-val:  {gap:.4f}  (objetivo < 0.05)')
print(f'KPI (>=0.90):      {"✓ CUMPLIDO" if val_acc_B >= 0.90 else "✗ NO cumplido"}')
print(f'Modelo final:      {MODELS_DIR}/rapidrelief_efficientnetb0_v5.keras')
print(f'TFLite INT8:       {MODELS_DIR}/rapidrelief_model.tflite')